# Evaluate the trained CCCNN

This notebook loads the saved checkpoint and grouped test split produced by the training notebook. It reports accuracy, per-class precision, recall and F1, macro-F1, ROC-AUC and average precision, plus confusion-matrix and curve plots.

Run the training notebook first. The model and split manifest are kept under artifacts/.


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image, ImageOps
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    precision_recall_fscore_support, roc_auc_score,
    average_precision_score, roc_curve, precision_recall_curve,
)
from sklearn.preprocessing import label_binarize
import matplotlib.pyplot as plt

def find_project_root(start):
    for candidate in (start, *start.parents):
        if (candidate / "artifacts").is_dir() and (candidate / "data_processed").is_dir():
            return candidate
    raise FileNotFoundError("Could not find artifacts/ and data_processed/. Run the training notebook first.")

PROJECT_ROOT = find_project_root(Path.cwd().resolve())
DATA_DIR = PROJECT_ROOT / "data_processed"
ARTIFACT_DIR = PROJECT_ROOT / "artifacts"
EVAL_DIR = ARTIFACT_DIR / "evaluation"
EVAL_DIR.mkdir(parents=True, exist_ok=True)
MODEL_PATH = ARTIFACT_DIR / "ccc_nn_best.pt"
SPLIT_PATH = ARTIFACT_DIR / "split_manifest.csv"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 64

if not MODEL_PATH.is_file() or not SPLIT_PATH.is_file():
    raise FileNotFoundError("Missing model or split manifest. Run the training notebook first.")
print("Project:", PROJECT_ROOT)
print("Device:", DEVICE)


FileNotFoundError: Could not find artifacts/ and data_processed/. Run the training notebook first.

## Load the held-out test set


In [ ]:
manifest = pd.read_csv(SPLIT_PATH)
test_frame = manifest.loc[manifest["split"] == "test"].reset_index(drop=True)
class_names = sorted(manifest["class_name"].unique())
class_to_id = {name: i for i, name in enumerate(class_names)}
y_true = test_frame["class_name"].map(class_to_id).to_numpy()

class TestDataset(Dataset):
    def __init__(self, frame):
        self.frame = frame
        self.to_tensor = transforms.ToTensor()

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, index):
        row = self.frame.iloc[index]
        path = PROJECT_ROOT / row["relative_path"]
        with Image.open(path) as image:
            image = ImageOps.exif_transpose(image).convert("RGB")
        return self.to_tensor(image), int(class_to_id[row["class_name"]])

test_loader = DataLoader(TestDataset(test_frame), batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
print("Test patches:", len(test_frame))
display(test_frame["class_name"].value_counts().reindex(class_names, fill_value=0).rename("patches").to_frame())


## Rebuild the CCCNN and load the saved weights


In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )

    def forward(self, x):
        return self.layers(x)

class CCCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.features = nn.Sequential(ConvBlock(3, 32), ConvBlock(32, 64), ConvBlock(64, 128))
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, 64),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.4),
            nn.Linear(64, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.pool(self.features(x)))

checkpoint = torch.load(MODEL_PATH, map_location=DEVICE, weights_only=True)
if checkpoint["class_names"] != class_names:
    raise ValueError("Class names in checkpoint and split manifest do not match.")
model = CCCNN(num_classes=len(class_names)).to(DEVICE)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()
print("Loaded best validation checkpoint.")


## Predict and calculate metrics


In [ ]:
all_probabilities = []
all_predictions = []
with torch.no_grad():
    for images, _ in test_loader:
        logits = model(images.to(DEVICE))
        probabilities = torch.softmax(logits, dim=1).cpu().numpy()
        all_probabilities.append(probabilities)
        all_predictions.extend(probabilities.argmax(axis=1))

y_pred = np.asarray(all_predictions)
y_prob = np.concatenate(all_probabilities, axis=0)

accuracy = accuracy_score(y_true, y_pred)
precision, recall, f1, support = precision_recall_fscore_support(
    y_true, y_pred, labels=np.arange(len(class_names)), zero_division=0
)
macro_f1 = float(np.mean(f1))
metrics_table = pd.DataFrame({
    "class": class_names,
    "precision": precision,
    "recall": recall,
    "f1": f1,
    "support": support,
})
display(metrics_table.round(4))
print(f"Accuracy: {accuracy:.4f}")
print(f"Macro-F1: {macro_f1:.4f}")
print()
print(classification_report(y_true, y_pred, labels=np.arange(len(class_names)), target_names=class_names, zero_division=0))

metrics_table.to_csv(EVAL_DIR / "per_class_metrics.csv", index=False)


## Confusion matrix


In [ ]:
cm = confusion_matrix(y_true, y_pred, labels=np.arange(len(class_names)))
plt.figure(figsize=(7, 6))
plt.imshow(cm, interpolation="nearest", cmap="Blues")
plt.colorbar()
plt.xticks(np.arange(len(class_names)), class_names, rotation=45, ha="right")
plt.yticks(np.arange(len(class_names)), class_names)
for row in range(cm.shape[0]):
    for col in range(cm.shape[1]):
        plt.text(col, row, str(cm[row, col]), ha="center", va="center", color="black")
plt.xlabel("Predicted class")
plt.ylabel("True class")
plt.title("Test confusion matrix")
plt.tight_layout()
plt.savefig(EVAL_DIR / "confusion_matrix.png", dpi=160)
plt.show()


## ROC and precision-recall curves

Curves are calculated one class at a time against all other classes. The average precision score summarizes the precision-recall curve; it is useful when classes are imbalanced.


In [ ]:
y_binary = label_binarize(y_true, classes=np.arange(len(class_names)))
curve_rows = []

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for class_id, class_name in enumerate(class_names):
    truth = y_binary[:, class_id]
    if truth.sum() == 0 or truth.sum() == len(truth):
        print(f"Skipping curves for {class_name}: test set needs positive and negative examples.")
        continue
    fpr, tpr, _ = roc_curve(truth, y_prob[:, class_id])
    precision_curve, recall_curve, _ = precision_recall_curve(truth, y_prob[:, class_id])
    auc_value = roc_auc_score(truth, y_prob[:, class_id])
    ap_value = average_precision_score(truth, y_prob[:, class_id])
    curve_rows.append({"class": class_name, "roc_auc": auc_value, "average_precision": ap_value})
    axes[0].plot(fpr, tpr, label=f"{class_name} (AUC={auc_value:.3f})")
    axes[1].plot(recall_curve, precision_curve, label=f"{class_name} (AP={ap_value:.3f})")

axes[0].plot([0, 1], [0, 1], linestyle="--", color="gray")
axes[0].set(title="One-vs-rest ROC curves", xlabel="False positive rate", ylabel="True positive rate")
axes[1].set(title="One-vs-rest precision-recall curves", xlabel="Recall", ylabel="Precision")
for ax in axes:
    ax.legend(fontsize=8)
    ax.grid(alpha=0.25)
plt.tight_layout()
plt.savefig(EVAL_DIR / "roc_pr_curves.png", dpi=160)
plt.show()

curve_table = pd.DataFrame(curve_rows)
display(curve_table.round(4))
if not curve_table.empty:
    print("Macro ROC-AUC:", round(curve_table["roc_auc"].mean(), 4))
    print("Macro average precision:", round(curve_table["average_precision"].mean(), 4))
curve_table.to_csv(EVAL_DIR / "curve_metrics.csv", index=False)


## Save a test prediction table

The CSV includes each held-out patch, its true and predicted labels, and the probability assigned to the predicted class.


In [ ]:
prediction_table = test_frame[["relative_path", "class_name"]].rename(columns={"class_name": "true_class"}).copy()
prediction_table["predicted_class"] = [class_names[i] for i in y_pred]
prediction_table["predicted_probability"] = y_prob.max(axis=1)
prediction_table.to_csv(EVAL_DIR / "test_predictions.csv", index=False)

summary = pd.DataFrame([{
    "test_patches": len(y_true),
    "accuracy": accuracy,
    "macro_f1": macro_f1,
    "macro_roc_auc": curve_table["roc_auc"].mean() if not curve_table.empty else np.nan,
    "macro_average_precision": curve_table["average_precision"].mean() if not curve_table.empty else np.nan,
}])
display(summary.round(4))
summary.to_csv(EVAL_DIR / "summary_metrics.csv", index=False)
print("Evaluation outputs saved in:", EVAL_DIR)
